# Manufacturing Data Analysis EDA Skeleton

## Project Context

This notebook is a minimal starting point for later exploratory data analysis and project presentation work.

Current goal:

- understand equipment, line, shift, quality, and failure behavior from the current processed manufacturing dataset
- keep the analysis aligned with the current repository structure and contract tests

Important boundary:

- several KPI fields in this project, including `planned_production`, `actual_production`, `defect_rate`, `availability`, `performance`, `quality_rate`, and `oee`, are current project proxy / simulated metrics
- this notebook should support structured analysis, but not claim industrial-grade plant conclusions at this stage


## Analysis Questions

This notebook is designed to help answer questions like:

1. Which equipment or production lines show weaker OEE performance?
2. Do `Day`, `Evening`, and `Night` shifts show different KPI patterns?
3. How does machine failure relate to quality and OEE behavior?
4. Which equipment appears more often in failure-labelled records?
5. Which insights are supported by current proxy metrics, and which conclusions are still outside the scope of this project?


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd


## Data Loading

Choose one processed CSV source.

- `manufacturing_data_processed.csv` is the current legacy reference
- `data/processed/manufacturing_data_processed_refactor.csv` is the current refactor snapshot

The default below now prefers the refactor snapshot, because it includes finer failure mode labels (`TWF`, `HDF`, `PWF`, `OSF`, `RNF`) used later in the notebook.


In [ ]:
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()

legacy_processed_path = PROJECT_ROOT / 'manufacturing_data_processed.csv'
refactor_processed_path = PROJECT_ROOT / 'data' / 'processed' / 'manufacturing_data_processed_refactor.csv'
failure_mode_columns = ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']

data_path = refactor_processed_path if refactor_processed_path.exists() else legacy_processed_path

print(f'Using dataset: {data_path}')
df = pd.read_csv(data_path)
available_failure_mode_columns = [column for column in failure_mode_columns if column in df.columns]

print(f'Shape: {df.shape}')
print(f'Available failure mode columns: {available_failure_mode_columns}')
display(df.head())


In [ ]:
df.columns.tolist()

## Dataset Overview

Start with the most basic checks before writing any interpretation.

This section answers a simple question first:

- What data do we actually have in hand before we compare equipment, lines, shifts, or failures?


### A. Basic Dataset Size And Key Column Presence

This check confirms whether the dataset has the expected scale and whether the main analysis fields are present.

In [ ]:
key_columns = ['production_time', 'equipment_id', 'oee', 'Machine failure']

basic_overview = {
    'row_count': len(df),
    'column_count': len(df.columns),
    'equipment_count': df['equipment_id'].nunique(),
    'production_line_count': df['production_line'].nunique(),
    'shift_categories': sorted(df['shift'].dropna().unique().tolist()),
    'key_columns_present': {column: (column in df.columns) for column in key_columns},
}

basic_overview

### B. Time Range Overview

This check tells us how wide the current analysis window is and how many calendar dates are covered.

In [ ]:
production_time_series = pd.to_datetime(df['production_time'])

time_overview = {
    'min_production_time': production_time_series.min(),
    'max_production_time': production_time_series.max(),
    'date_count': production_time_series.dt.date.nunique(),
}

time_overview

### C. Basic Missing Check

This is not a full data-quality audit. It only checks whether the most important fields are missing obvious values.

In [ ]:
missing_check_columns = [
    'production_time',
    'equipment_id',
    'production_line',
    'shift',
    'oee',
    'availability',
    'performance',
    'quality_rate',
    'Machine failure',
]

missing_summary = df[missing_check_columns].isna().sum().rename('missing_count').to_frame()
missing_summary['missing_ratio'] = (missing_summary['missing_count'] / len(df)).round(6)
missing_summary

### D. Basic KPI Summary

This gives a first numeric profile of the current KPI fields.

Important reminder:

- these are current project proxy / simulated metrics
- treat this as a local analysis baseline, not as industrial KPI certification


In [ ]:
kpi_summary = df[['oee', 'availability', 'performance', 'quality_rate']].describe().T
kpi_summary

### E. Short Interpretation Notes

Use this area to write 2 to 4 short observations after you inspect the outputs above.

Suggested prompts:

- Does the dataset size match what you expected?
- Are the key analysis columns present and mostly complete?
- Does the current time range look reasonable for a first analysis pass?
- Do the KPI summaries look broadly plausible under the current proxy logic?


## Equipment / Line KPI Analysis

Use this section to compare equipment and production lines.

This section answers the next practical question:

- Which equipment or lines look weaker under the current KPI rules, and where should deeper analysis start?


### A. Equipment-Level KPI Summary

This table is the first place to look for weaker equipment under the current proxy KPI design.

Suggested reading habit:

- first scan low `avg_oee`
- then check whether low `avg_oee` appears together with higher `failure_record_count` or `total_defect_count`


In [ ]:
equipment_kpi = (
    df.groupby('equipment_id', as_index=False)
    .agg(
        production_line=('production_line', 'first'),
        avg_oee=('oee', 'mean'),
        avg_availability=('availability', 'mean'),
        avg_performance=('performance', 'mean'),
        avg_quality_rate=('quality_rate', 'mean'),
        total_actual_production=('actual_production', 'sum'),
        total_defect_count=('defect_count', 'sum'),
        failure_record_count=('Machine failure', 'sum'),
    )
    .sort_values(['avg_oee', 'failure_record_count', 'total_defect_count'], ascending=[True, False, False])
    .reset_index(drop=True)
)

equipment_kpi.head(10)

Interpretation prompt:

- Which equipment sits near the bottom of `avg_oee`?
- Are those same equipment IDs also showing more failure-labelled records or more total defects?
- Under the current proxy KPI rules, this supports prioritizing inspection targets, but it does not prove root cause.


### B. Production-Line KPI Summary

This table rolls the same logic up to production-line level so you can see whether weaker equipment also cluster into weaker lines.

In [ ]:
line_kpi = (
    df.groupby('production_line', as_index=False)
    .agg(
        avg_oee=('oee', 'mean'),
        avg_availability=('availability', 'mean'),
        avg_performance=('performance', 'mean'),
        avg_quality_rate=('quality_rate', 'mean'),
        total_actual_production=('actual_production', 'sum'),
        total_defect_count=('defect_count', 'sum'),
        failure_record_count=('Machine failure', 'sum'),
    )
    .sort_values(['avg_oee', 'failure_record_count'], ascending=[True, False])
    .reset_index(drop=True)
)

line_kpi

Interpretation prompt:

- Which production line looks weakest on average `oee`?
- Does the weaker line also show more failure-labelled records or more defects?
- Under the current project scope, this supports line-level comparison, but not final industrial performance judgment.


### C. Simple Ranking Views

These quick ranking tables help you identify where to focus next without introducing charts yet.

In [ ]:
lowest_avg_oee_equipment = equipment_kpi.nsmallest(5, 'avg_oee')
highest_defect_equipment = equipment_kpi.nlargest(5, 'total_defect_count')
line_ranking = line_kpi.sort_values(['avg_oee', 'failure_record_count'], ascending=[True, False])

display(lowest_avg_oee_equipment)
display(highest_defect_equipment)
display(line_ranking)

Interpretation prompt:

- Do the weakest `avg_oee` equipment match the highest-defect equipment?
- Are the same lines repeatedly appearing at the weaker end of the ranking?
- These rankings are good starting points for later shift and failure analysis, but they are still descriptive, not causal.


## Shift KPI Analysis

Use this section to compare `Day`, `Evening`, and `Night` shifts.

This section answers the next practical question:

- Which shifts look weaker under the current KPI rules, and do weaker shifts also appear together with more failures or defects?


### A. Shift-Level KPI Summary

This table is the first place to compare `Day`, `Evening`, and `Night` at a high level.

In [ ]:
shift_kpi = (
    df.groupby('shift', as_index=False)
    .agg(
        avg_oee=('oee', 'mean'),
        avg_availability=('availability', 'mean'),
        avg_performance=('performance', 'mean'),
        avg_quality_rate=('quality_rate', 'mean'),
        total_actual_production=('actual_production', 'sum'),
        total_defect_count=('defect_count', 'sum'),
        failure_record_count=('Machine failure', 'sum'),
    )
    .sort_values(['avg_oee', 'failure_record_count'], ascending=[True, False])
    .reset_index(drop=True)
)

shift_kpi

Interpretation prompt:

- Which shift sits at the bottom of average `oee`?
- Does the weaker shift also show more failure-labelled records or more defects?
- Under the current proxy KPI logic, this supports shift comparison, but it does not prove staffing, scheduling, or operational root cause.


### B. Production-Line + Shift Summary

This table helps answer a more practical follow-up question:

- Is a shift weak everywhere, or only weak inside specific production lines?


In [ ]:
line_shift_kpi = (
    df.groupby(['production_line', 'shift'], as_index=False)
    .agg(
        avg_oee=('oee', 'mean'),
        avg_availability=('availability', 'mean'),
        avg_performance=('performance', 'mean'),
        avg_quality_rate=('quality_rate', 'mean'),
        total_actual_production=('actual_production', 'sum'),
        total_defect_count=('defect_count', 'sum'),
        failure_record_count=('Machine failure', 'sum'),
    )
    .sort_values(['avg_oee', 'failure_record_count', 'total_defect_count'], ascending=[True, False, False])
    .reset_index(drop=True)
)

line_shift_kpi

Interpretation prompt:

- Does a weak shift appear in all lines or mainly in one line?
- Which line-shift combinations deserve a closer look next?
- This is useful for targeted follow-up, but it is still descriptive under the current simulated metric design.


### C. Simple Ranking Views

These ranking tables make it easier to spot weaker shifts without adding charts yet.

In [ ]:
lowest_avg_oee_shift = shift_kpi.nsmallest(3, 'avg_oee')
highest_failure_shift = shift_kpi.nlargest(3, 'failure_record_count')
line_shift_ranking = line_shift_kpi.sort_values(['avg_oee', 'failure_record_count'], ascending=[True, False])

display(lowest_avg_oee_shift)
display(highest_failure_shift)
display(line_shift_ranking)


Interpretation prompt:

- Are the weakest `avg_oee` shifts also the same shifts with the most failure-labelled records?
- Which line-shift combinations appear repeatedly near the weaker end of the ranking?
- These rankings help prioritize deeper inspection, but they do not support industrial-grade conclusions about staffing, maintenance, or scheduling by themselves.


## Failure Summary

Use this section to inspect failure-labelled records and compare them with non-failure records.

This section now uses both the broader `Machine failure` label and the finer AI4I failure mode labels when they are available in the refactor snapshot.

This section answers the next practical question:

- How different do failure-labelled records look under the current proxy KPI rules, which failure modes appear most often, and where should deeper failure analysis start?


### A. Machine Failure vs Non-Failure Comparison

This table is the first place to compare broad failure-labelled and non-failure records under the current project logic.

Suggested reading habit:

- first compare `avg_oee` and `avg_quality_rate`
- then check whether failure-labelled records also show higher `avg_defect_rate` or `total_defect_count`


In [ ]:
machine_failure_summary = (
    df.groupby('Machine failure', as_index=False)
    .agg(
        avg_oee=('oee', 'mean'),
        avg_availability=('availability', 'mean'),
        avg_performance=('performance', 'mean'),
        avg_quality_rate=('quality_rate', 'mean'),
        avg_defect_rate=('defect_rate', 'mean'),
        total_actual_production=('actual_production', 'sum'),
        total_defect_count=('defect_count', 'sum'),
        record_count=('UDI', 'count'),
    )
    .sort_values('Machine failure', ascending=False)
    .reset_index(drop=True)
)

machine_failure_summary


Interpretation prompt:

- Do failure-labelled records show lower `avg_oee`, `avg_availability`, or `avg_quality_rate` than non-failure records?
- Do they also show higher `avg_defect_rate` or more total defects?
- Under the current proxy KPI rules, this supports descriptive comparison, but it does not prove causal failure mechanisms.


### B. Failure Mode Frequency Summary

This table shows which finer failure mode labels appear more often in the current refactor snapshot.


In [ ]:
failure_mode_frequency_summary = pd.DataFrame([
    {
        'failure_mode': column,
        'positive_count': int(df[column].sum()),
        'positive_rate': float(df[column].mean()),
    }
    for column in available_failure_mode_columns
]).sort_values(['positive_count', 'failure_mode'], ascending=[False, True]).reset_index(drop=True)

failure_mode_frequency_summary


Interpretation prompt:

- Which failure mode appears most often in the current dataset?
- Are some failure modes rare enough that later analysis should treat them carefully?
- These counts describe the current snapshot only. They do not prove plant-level prevalence outside this dataset.


### C. Failure Mode KPI Comparison

This table compares KPI behavior inside each finer failure mode by filtering rows where that mode equals `1`.


In [ ]:
failure_mode_kpi_summary = pd.DataFrame([
    {
        'failure_mode': column,
        'avg_oee': df.loc[df[column] == 1, 'oee'].mean(),
        'avg_quality_rate': df.loc[df[column] == 1, 'quality_rate'].mean(),
        'avg_defect_rate': df.loc[df[column] == 1, 'defect_rate'].mean(),
        'total_defect_count': int(df.loc[df[column] == 1, 'defect_count'].sum()),
        'record_count': int((df[column] == 1).sum()),
    }
    for column in available_failure_mode_columns
]).sort_values(['record_count', 'avg_oee'], ascending=[False, True]).reset_index(drop=True)

failure_mode_kpi_summary


Interpretation prompt:

- Which failure mode is associated with lower `avg_oee` or lower `avg_quality_rate` under the current project logic?
- Which failure mode is associated with more total defects?
- These comparisons help prioritize follow-up analysis, but they do not prove industrial root cause.


### D. Equipment Failure Mode Entry Point

This table gives a first equipment-level entry point for checking which equipment IDs appear more often under different failure modes.


In [ ]:
equipment_failure_mode_entry = (
    df.groupby(['equipment_id', 'production_line'], as_index=False)[available_failure_mode_columns]
    .sum()
    .sort_values(available_failure_mode_columns, ascending=[False] * len(available_failure_mode_columns))
    .reset_index(drop=True)
)

equipment_failure_mode_entry.head(10)


Interpretation prompt:

- Do some equipment IDs appear more often under a specific failure mode label?
- Are those equipment IDs already familiar from weaker OEE or higher-defect summaries earlier in the notebook?
- Under the current project scope, this is a useful investigation entry point, not a final maintenance diagnosis.


### E. Simple Failure Ranking Views

These quick ranking tables help you decide what to inspect next without adding charts yet.


In [ ]:
top_failure_modes = failure_mode_frequency_summary.head(5)
top_failure_mode_equipment = equipment_failure_mode_entry.head(10)

display(machine_failure_summary)
display(top_failure_modes)
display(top_failure_mode_equipment)


Interpretation prompt:

- Which failure mode should be explored first because it appears most often?
- Which equipment IDs are the strongest candidates for mode-specific follow-up?
- These rankings support descriptive prioritization only. They do not prove causality, root cause, or industrial corrective action.


## Minimal Visuals

Use this section to turn a few of the most important ranking tables into lightweight presentation-ready visuals.

These charts stay intentionally simple:

- they reuse the summary tables already created above
- they support explanation, not advanced dashboarding
- they still rely on proxy / simulated metrics and should be read as descriptive visuals only


### A. Equipment Average OEE Ranking

This chart helps answer a simple manufacturing question first: which equipment IDs look weaker on average OEE and deserve earlier follow-up review?


In [ ]:
equipment_oee_plot = equipment_kpi.sort_values('avg_oee', ascending=True).head(8)

plt.figure(figsize=(8, 4))
plt.bar(equipment_oee_plot['equipment_id'], equipment_oee_plot['avg_oee'])
plt.title('Lowest Average OEE Equipment')
plt.xlabel('Equipment ID')
plt.ylabel('Average OEE')
plt.ylim(0, 1)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

equipment_oee_plot


Interpretation note:

This chart highlights which equipment IDs sit near the bottom of average OEE under the current project logic. It supports inspection prioritization, but it does not prove why those equipment IDs look weaker.


### B. Shift KPI Comparison

This chart compares shift-level `avg_oee` and `avg_quality_rate` so you can quickly see whether one shift looks weaker than the others.


In [ ]:
shift_plot = shift_kpi.sort_values('avg_oee', ascending=True).reset_index(drop=True)
shift_positions = range(len(shift_plot))
bar_width = 0.35

plt.figure(figsize=(7, 4))
plt.bar([position - bar_width / 2 for position in shift_positions], shift_plot['avg_oee'], width=bar_width, label='avg_oee')
plt.bar([position + bar_width / 2 for position in shift_positions], shift_plot['avg_quality_rate'], width=bar_width, label='avg_quality_rate')
plt.title('Shift KPI Comparison')
plt.xlabel('Shift')
plt.ylabel('Score')
plt.ylim(0, 1)
plt.xticks(list(shift_positions), shift_plot['shift'])
plt.legend()
plt.tight_layout()
plt.show()

shift_plot[['shift', 'avg_oee', 'avg_quality_rate', 'failure_record_count']]


Interpretation note:

This chart is a quick way to compare broad shift performance. It can highlight weaker shifts, but it still reflects descriptive notebook-level comparison rather than a proven operational cause.


### C. Failure Mode Frequency

This chart shows which finer failure mode labels appear most often in the current refactor snapshot.


In [ ]:
failure_mode_plot = failure_mode_frequency_summary.sort_values('positive_count', ascending=False).reset_index(drop=True)

plt.figure(figsize=(7, 4))
plt.bar(failure_mode_plot['failure_mode'], failure_mode_plot['positive_count'])
plt.title('Failure Mode Frequency')
plt.xlabel('Failure Mode')
plt.ylabel('Positive Record Count')
plt.tight_layout()
plt.show()

failure_mode_plot


Interpretation note:

This chart helps show which finer failure mode labels are more common in the current snapshot. It is useful for prioritizing later failure analysis, but it should not be treated as a plant-wide prevalence conclusion.


## Findings Draft

Use this section to turn the tables above into short, honest, first-pass findings.

Important boundary for every draft below:

- these findings are descriptive, not causal
- current KPI fields such as `oee`, `quality_rate`, and `defect_rate` are proxy / simulated metrics
- any stronger industrial interpretation should be treated as *to be confirmed* rather than final proof

### Finding Draft 1: Equipment / Line KPI Angle

- Observation:
  Equipment IDs near the bottom of the `equipment_kpi` ranking, and production lines near the bottom of `line_kpi`, look like the first candidates for follow-up review. The exact weakest equipment or line is *to be confirmed from the ranking table above*.
- Evidence:
  Use `equipment_kpi`, `lowest_avg_oee_equipment`, `highest_defect_equipment`, and `line_ranking` from the Equipment / Line KPI section.
- Business meaning:
  This helps narrow the first inspection scope to a smaller set of equipment and lines instead of treating the whole dataset as equally risky.
- Current limitation:
  Lower average `oee` or higher defect totals here do not prove a root cause. They only identify where deeper review should begin under the current simulated metric logic.

### Finding Draft 2: Shift Angle

- Observation:
  One or more shifts may show weaker KPI patterns, and some `production_line + shift` combinations may stand out as more fragile than others. The exact priority combination is *to be confirmed from the shift ranking tables above*.
- Evidence:
  Use `shift_kpi`, `lowest_avg_oee_shift`, `highest_failure_shift`, and `line_shift_ranking` from the Shift KPI Analysis section.
- Business meaning:
  This can guide later review toward operational windows where process stability or output quality looks weaker on average.
- Current limitation:
  A weaker shift pattern in this notebook does not prove that the shift itself is the cause. It may still reflect the current synthetic time logic, equipment mix, or other unmodeled factors.

### Finding Draft 3: Failure Angle

- Observation:
  Failure-labelled records appear to differ from non-failure records, and some finer failure mode labels such as `HDF`, `OSF`, `PWF`, `TWF`, or `RNF` may be more common than others in the current snapshot. The strongest mode-specific takeaway is *to be confirmed from the failure summaries above*.
- Evidence:
  Use `machine_failure_summary`, `failure_mode_frequency_summary`, `failure_mode_kpi_summary`, and `equipment_failure_mode_entry` from the Failure Summary section.
- Business meaning:
  This gives a first failure-focused prioritization path: compare broad failure vs non-failure behavior, then check which finer failure modes and equipment IDs deserve closer inspection.
- Current limitation:
  These results do not prove industrial failure causality or fault diagnosis. The failure mode labels are useful for structured comparison, but they are still being analyzed on top of proxy / simulated KPI fields.


### Next Analysis Focus

A few directions are the best candidates for the next round of charts or deeper explanation:

- compare the bottom-ranked equipment against the bottom-ranked line to see whether weak equipment cluster in one line
- compare the weakest shift and the most fragile `line + shift` combinations before deciding which shift visuals are worth building
- visualize the most common failure mode labels and the equipment IDs most often associated with them


## Limitations

Keep these limits visible when writing conclusions:

- current OEE, quality, and production fields are proxy / simulated metrics under current project rules
- current timestamps, lines, and equipment identifiers are partly generated by the preprocessing logic
- this notebook supports structured local analysis, not industrial-grade operational claims
- later notebook and dashboard work should stay aligned with `docs/analysis_report_template.md`
